# Aerial Fire Vision: first dataset check

Run the cells **top to bottom in Google Colab**. This will **not unzip or modify** the original archives.

1. Connect to the shared folder.
2. Verify all four ZIP inventories.
3. Preview a few classification examples.
4. Check original image/mask pairing and an overlay.

Add a shortcut to **Vision AI Project** from Google Drive's **Shared with me** into **My Drive** before starting. Do not print unrelated personal Drive filenames.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")

# Update this path only if you placed the shortcut elsewhere.
DATA_DIR = Path("/content/drive/MyDrive/Vision AI Project")
if not DATA_DIR.is_dir():
    raise FileNotFoundError("Cannot find shared folder. Add a shortcut to My Drive or update DATA_DIR.")
print("Shared folder found:", DATA_DIR)

## 1. ZIP inventory (no extraction)

Expected image counts: **Training 39,375; Test 8,617; Images 2,003; Masks 2,003**. Note that **Test.zip** is singular. Listing files does not guarantee they all decode correctly.

In [ ]:
from zipfile import ZipFile
from pathlib import PurePosixPath
from collections import Counter

EXPECTED = {"Training": 39375, "Test": 8617, "Images": 2003, "Masks": 2003}
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}
archives, files = {}, {}
all_zips = list(DATA_DIR.rglob("*.zip"))

for name, expected_count in EXPECTED.items():
    found = [p for p in all_zips if p.stem.lower() == name.lower()]
    if len(found) != 1:
        print(f"CHECK: {name}.zip has {len(found)} matches, expected 1")
        continue
    archives[name] = found[0]
    try:
        with ZipFile(found[0]) as z:
            files[name] = [n for n in z.namelist()
                           if PurePosixPath(n).suffix.lower() in IMAGE_EXT
                           and "__MACOSX" not in PurePosixPath(n).parts]
    except Exception as exc:
        print(f"ERROR: cannot read {name}.zip: {exc}")
        archives.pop(name)
        continue
    count = len(files[name])
    print(f"{'OK' if count == expected_count else 'CHECK'}: {name}.zip {count:,} / {expected_count:,}")
    if name in ("Training", "Test"):
        counts = Counter()
        for n in files[name]:
            parts = PurePosixPath(n).parts
            label = next((parts[i+1] for i,p in enumerate(parts[:-1])
                          if p.lower() == name.lower()), "Unknown")
            counts[label] += 1
        print("  Classes:", dict(counts))

## 2. Preview Fire / No_Fire classification images

We sample four images, **not all 47,992 classification frames**. No model training yet.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import random

def read_image(archive, member, mode="RGB"):
    with ZipFile(archives[archive]) as z:
        with z.open(member) as f:
            with Image.open(f) as img:
                return img.convert(mode)

rng = random.Random(42)
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for row, split in enumerate(["Training", "Test"]):
    for col, label in enumerate(["Fire", "No_Fire"]):
        ax = axes[row, col]
        options = [f for f in files.get(split, [])
                   if label.lower() in [x.lower() for x in PurePosixPath(f).parts]]
        if options:
            img = read_image(split, rng.choice(options))
            ax.imshow(img)
            ax.set_title(f"{split} / {label} ({img.width}x{img.height})")
        else:
            ax.set_title(f"Missing: {split}/{label}")
        ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Verify Images ↔ Masks (segmentation)

**Do not match Training with Masks:** they belong to different FLAME tasks. The masks correspond to **Images.zip** and mark fire pixels. This displays only one matched image, mask, and red overlay.

In [ ]:
import numpy as np

if "Images" not in archives or "Masks" not in archives:
    print("Need Images.zip and Masks.zip to inspect segmentation.")
else:
    image_index = {PurePosixPath(f).stem.lower(): f for f in files["Images"]}
    mask_index = {PurePosixPath(f).stem.lower(): f for f in files["Masks"]}
    paired = sorted(image_index.keys() & mask_index.keys())
    print(f"Matching pairs: {len(paired):,} / 2,003 expected")
    print("Images without masks:", len(image_index) - len(paired))
    print("Masks without images:", len(mask_index) - len(paired))
    if paired:
        key = rng.choice(paired)
        image = read_image("Images", image_index[key])
        mask = read_image("Masks", mask_index[key], mode="L")
        print("Example:", key, "Image:", image.size, "Mask:", mask.size)
        print("Matching dimensions:", image.size == mask.size)
        print("Mask values:", np.unique(np.asarray(mask))[:20])

        small = image.copy()
        small.thumbnail((850, 650))
        mask_small = mask.resize(small.size, Image.Resampling.NEAREST)
        rgba = np.zeros((small.height, small.width, 4), dtype=np.uint8)
        rgba[np.asarray(mask_small) > 0] = [255, 30, 30, 145]
        fig, ax = plt.subplots(1, 3, figsize=(15, 5))
        ax[0].imshow(small); ax[0].set_title("Original")
        ax[1].imshow(mask_small, cmap="gray"); ax[1].set_title("Fire mask")
        ax[2].imshow(small); ax[2].imshow(rgba); ax[2].set_title("Mask overlay")
        for a in ax: a.axis("off")
        plt.tight_layout()
        plt.show()

## What to tell the team

**Wednesday Nov 11 (~7 PM, email):** Can you access all four archives? Any blockers?

**Monday Nov 16 (meeting):** Show the counts, image-mask pairing, a couple of images, and any small method experiment or observations. **No need to process 52,000 files or train a large model this week.**

This is a *sample-access audit*, not a full corruption scan. The dataset's Fire/No_Fire classification task is **visible flames**, not verified smoke detection. Do not tune models on the held-out Phantom test set. Upload only small code/notebook changes through GitHub pull requests; do not commit images or personal Drive listings.